In [112]:
import torch


In [113]:
a = torch.arange(24)
b = a.reshape(2,3,4)
b

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])

In [114]:
c = b[1:2]
c

tensor([[[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])

In [115]:
print(f"{a.storage_offset()=}")
print(f"{c.storage_offset()=}")
print(f"{a.stride()=}")
print(f"{b.stride()=}")
print(f"{a.data_ptr()=}")
print(f"{b.data_ptr()=}")
print(f"{a.device=}")
print(f"{b.device=}")

a.storage_offset()=0
c.storage_offset()=12
a.stride()=(1,)
b.stride()=(12, 4, 1)
a.data_ptr()=4834119977088
b.data_ptr()=4834119977088
a.device=device(type='cpu')
b.device=device(type='cpu')


In [116]:
c = a.reshape(4,6)
c = c.t()

In [117]:
print(f"{a.data_ptr()=}")
print(f"{c.data_ptr()=}")
# print()

d = c.contiguous() # => returns copy since this data can't be arranged as continguous view
print(f"{d.data_ptr()=}")
print(d.stride())
print(c.stride())

a.data_ptr()=4834119977088
c.data_ptr()=4834119977088
d.data_ptr()=4834119977280
(4, 1)
(1, 6)


In [118]:
print(f"{c.is_contiguous()=}")
c.view(-1) # asking for a 1 d view , which not possible, but a reshape copies data and makes it possible

c.is_contiguous()=False


RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [119]:
e = c.reshape(-1)
print(e)
print(f"{e.data_ptr()=}")

tensor([ 0,  6, 12, 18,  1,  7, 13, 19,  2,  8, 14, 20,  3,  9, 15, 21,  4, 10,
        16, 22,  5, 11, 17, 23])
e.data_ptr()=4834119977472


In [120]:
x = torch.arange(4)
y = x.expand((3,4)) # broadcasting (sets stride = 0 for row)
print(y)
print(f"{x.shape=}")
print(f"{y.shape=}")
print(f"{x.stride()=}")
print(f"{y.stride()=}")

tensor([[0, 1, 2, 3],
        [0, 1, 2, 3],
        [0, 1, 2, 3]])
x.shape=torch.Size([4])
y.shape=torch.Size([3, 4])
x.stride()=(1,)
y.stride()=(0, 1)


In [121]:
x = x.to("cuda")
#by defalut tensors have requres grad false
y = torch.ones_like(x, device= "cuda", requires_grad= True, dtype= torch.float32)
z = x  * y; z.retain_grad() # by defalut intermediate tensors don't retain grad, intermediate 
                            #intermediate tensors produced by op a tensor that has req grad true will also have req grad true
l = z.sum()
l.backward()

In [137]:
try:
    l.backward()
except Exception as e:
    print(e)

Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.


In [122]:
print(f"{z.grad=}")
print(f"{y.grad=}")
print(f"{x.grad=}")


z.grad=tensor([1., 1., 1., 1.], device='cuda:0')
y.grad=tensor([0., 1., 2., 3.], device='cuda:0')
x.grad=None


In [123]:
print(f"{y.grad_fn=}")
print(f"{z.grad_fn=}")

y.grad_fn=None
z.grad_fn=<MulBackward0 object at 0x0000020DE1D06CE0>


In [124]:
#tensor with  gradfn  = none is considered a leaf tensor
print(f"{z.is_leaf=}")
print(f"{y.is_leaf=}")


z.is_leaf=False
y.is_leaf=True


In [133]:
a = torch.tensor([1,2,3])
print(f"{a.device=}")
b = a.cuda()
print(f"{b.device=}")
print(b)

a.device=device(type='cpu')
b.device=device(type='cuda', index=0)
tensor([1, 2, 3], device='cuda:0')


In [141]:
t = torch.arange(12).reshape(3, 4)
t.T.view(12)                 # RuntimeError: view size is not compatible...
t.T.reshape(12)              # works — reshape() falls back to a copy when needed
t.T.contiguous().view(12)    # also works — contiguous() materializes a fresh copy first

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [391]:
class Model():
    def __init__(self, nin, nout):
        self.W = torch.randn((nin, nout), requires_grad = True)
        self.B = torch.randn((1, nout), requires_grad = True)

    def __call__(self, data_batch, inference = False):
        if inference:
            with torch.no_grad():
                return (data_batch @ self.W) + self.B
        print(f"{data_batch.shape=}")
        print(f"{self.W.shape=}")
        print(f"{self.B.shape=}")
        return (data_batch @ self.W) + self.B

    def loss(self, data_batch, y_true):
        print(f"{y_true.shape=}")
        return torch.mean(torch.square(y_true - self(data_batch))) #don't use inplace operations it will cause autograd and backprop erorrs
                                                                    #let it generate new intermediate activations
    def update(self, lr):
        with torch.no_grad():
            self.W -=  lr * self.W.grad        #in place operataion better not to record these operations (anyhow no use)
            self.B -=  lr * self.B.grad
        self.W.grad.zero_()
        self.B.grad.zero_()
            

In [392]:
X =  torch.rand((50, 4)) * 100
Y =  X @ (torch.tensor([2, 4, 8, 1.0]).reshape(4,1))  + 10

In [393]:
print(X[:10])
print(Y[:10])

tensor([[3.0766e+01, 5.3966e+01, 2.7470e+00, 1.4514e-02],
        [3.9107e+01, 4.0401e+00, 6.6790e+01, 4.1025e+01],
        [8.8316e+01, 4.1574e+01, 9.0156e+01, 7.8467e+01],
        [1.1577e+01, 1.3981e+01, 3.2515e+00, 5.7154e+01],
        [6.1443e+01, 3.6298e+01, 6.0355e+01, 2.4630e+00],
        [4.0679e+01, 8.6490e+01, 3.7783e+01, 7.6027e+01],
        [7.1470e+01, 2.5715e+01, 3.2346e+01, 5.0902e+01],
        [9.4842e-01, 6.8767e+01, 5.3485e+01, 4.0093e+01],
        [6.9301e+01, 8.9567e+01, 1.2794e+00, 2.4434e+01],
        [8.9439e+01, 4.9279e+01, 9.6687e+01, 4.5052e+01]])
tensor([[ 309.3860],
        [ 679.7170],
        [1152.6453],
        [ 172.2423],
        [ 763.3798],
        [ 815.6106],
        [ 565.4712],
        [ 754.9381],
        [ 541.5398],
        [1204.5405]])


In [394]:
model = Model(4, 1)

In [395]:
model(X, inference=True)[:10]

tensor([[ 63.8260],
        [155.8497],
        [254.2316],
        [ 39.0933],
        [165.6604],
        [190.8093],
        [110.8226],
        [193.9020],
        [105.2130],
        [266.3489]])

In [396]:
loss = model.loss(X, Y)
print(f"{loss=}")
print("W\n", model.W)
print("B\n", model.B)
loss.backward()
print("W:grad\n", model.W.grad)
model.update(lr = 0.001)

y_true.shape=torch.Size([50, 1])
data_batch.shape=torch.Size([50, 4])
self.W.shape=torch.Size([4, 1])
self.B.shape=torch.Size([1, 1])
loss=tensor(335482.3125, grad_fn=<MeanBackward0>)
W
 tensor([[0.0250],
        [1.0297],
        [2.0584],
        [0.2779]], requires_grad=True)
B
 tensor([[1.8306]], requires_grad=True)
W:grad
 tensor([[-49507.8633],
        [-61120.2969],
        [-57545.3438],
        [-56815.3750]])


In [397]:
p = torch.randn((3, 4), requires_grad = True)
q = torch.randn((4,3))


In [398]:
r = p @ q
s = r.sum()
print(f"{s=}")

s=tensor(-13.9531, grad_fn=<SumBackward0>)


In [399]:
s.backward()

In [400]:
print(f"{p.grad=}")
print(f"{p.grad.shape=}")

p.grad=tensor([[-2.6726, -2.2604, -1.4090,  1.5688],
        [-2.6726, -2.2604, -1.4090,  1.5688],
        [-2.6726, -2.2604, -1.4090,  1.5688]])
p.grad.shape=torch.Size([3, 4])


In [401]:
print(p.data.data_ptr())
print(p.data_ptr())
p is p.data # false because underlying storage is same but pyton tensor object is differt (view)


4834120239872
4834120239872


False

In [402]:
n = torch.tensor([1, 2, 3], device= "cuda")
n.data  # n.data will be on same device as n


tensor([1, 2, 3], device='cuda:0')